In [1]:
#=========================================================================
#📘 FEATURE_EXTRACTION.PY (OPTIMIZADO PARA GITHUB)
#=========================================================================

#🎯 OBJETIVO:
#------------
#Cargar el dataset preprocesado (archivos .npz) y extraer embeddings 
#(vectores) por imagen usando MobileNetV2 preentrenada (sin la última 
#capa). Guardar los embeddings OPTIMIZADOS en embeddings.pkl

#✅ OPTIMIZACIONES IMPLEMENTADAS:
#---------------------------------
#1. Conversión a float16 (reduce 50% el tamaño)
#2. Reducción PCA opcional (reduce 80% más)
#3. Total: 50-85% de reducción según configuración

#📄 TECNOLOGÍA:
#-------------
#✅ MobileNetV2 (1280 dimensiones, 2018)
#✅ Compatible con TensorFlow 2.20+ y Keras 3.10+
#✅ Compatible con Python 3.13
#✅ Optimizado para GitHub (archivo <100MB)


In [2]:
# =========================================================================
# 📦 IMPORTAR LIBRERÍAS NECESARIAS
# =========================================================================

import os
import sys
import numpy as np
from tqdm import tqdm
import pickle
from pathlib import Path
import gc
from glob import glob
import warnings

# TensorFlow y Keras
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# ✅ Para reducción de dimensionalidad
from sklearn.decomposition import PCA

# Para cargar imágenes
from PIL import Image

# Visualización
import matplotlib.pyplot as plt

# Suprimir warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print("✅ Librerías importadas correctamente!")
print(f"🔧 TensorFlow versión: {tf.__version__}")
print(f"🔧 Keras backend: {tf.keras.backend.backend()}")
print(f"🧠 Modelo seleccionado: MobileNetV2")

✅ Librerías importadas correctamente!
🔧 TensorFlow versión: 2.20.0
🔧 Keras backend: tensorflow
🧠 Modelo seleccionado: MobileNetV2


In [3]:
# =========================================================================
# ⚙️ CONFIGURACIÓN DE PARÁMETROS
# =========================================================================

class Config:
    """Clase de configuración para el proceso de extracción"""
    
    # --- RUTAS ---
    DATASET_PATH = "../data/desayuno_preprocessed"
    OUTPUT_PATH = "../backend/models"
    OUTPUT_FILE = "embeddings.pkl"  # ✅ ARCHIVO .PKL (sin gzip)
    
    # --- PARÁMETROS DE MODELO ---
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 32
    EMBEDDING_DIM = 1280
    
    # --- ✅ PARÁMETROS DE OPTIMIZACIÓN ---
    USE_PCA = False              # True = Reducir a 256 dims (~85% reducción)
    PCA_COMPONENTS = 256         # Número de dimensiones después de PCA
    USE_FLOAT16 = True           # True = float16 (~50% reducción)
    
    # --- VISUALIZACIÓN ---
    GENERATE_PLOTS = True        # Generar gráficos de análisis

# Crear carpeta de salida
os.makedirs(Config.OUTPUT_PATH, exist_ok=True)

print(f"\n📂 Dataset: {Config.DATASET_PATH}")
print(f"💾 Salida: {os.path.join(Config.OUTPUT_PATH, Config.OUTPUT_FILE)}")
print(f"📦 Batch size: {Config.BATCH_SIZE}")
print(f"📏 Dimensiones embedding: {Config.EMBEDDING_DIM}")

print("\n🔧 CONFIGURACIÓN DE OPTIMIZACIÓN:")
print("=" * 34)
print(f"✅ Precisión reducida: float16 (ahorra ~50%)")
print(f"⚙️ PCA activado: {Config.USE_PCA} (si True, ahorra 80% más)")
reduction = 50 if not Config.USE_PCA else 85
print(f"🎯 Reducción esperada: ~{reduction}%")


📂 Dataset: ../data/desayuno_preprocessed
💾 Salida: ../backend/models\embeddings.pkl
📦 Batch size: 32
📏 Dimensiones embedding: 1280

🔧 CONFIGURACIÓN DE OPTIMIZACIÓN:
✅ Precisión reducida: float16 (ahorra ~50%)
⚙️ PCA activado: False (si True, ahorra 80% más)
🎯 Reducción esperada: ~50%


In [4]:
# =========================================================================
# 🔧 FUNCIONES AUXILIARES
# =========================================================================

def load_npz_files(dataset_path):
    """
    Cargar todos los archivos .npz del dataset
    
    Args:
        dataset_path (str): Ruta al directorio con archivos .npz
        
    Returns:
        list: Lista de rutas a archivos .npz
    """
    dataset_path = Path(dataset_path)
    npz_files = sorted(list(dataset_path.glob("*.npz")))
    
    print(f"\n📦 Archivos .npz encontrados: {len(npz_files)}")
    
    if len(npz_files) == 0:
        raise FileNotFoundError(
            f"❌ No se encontraron archivos .npz en {dataset_path}"
        )
    
    print(f"📋 Primeros archivos:")
    for f in npz_files[:5]:
        print(f"   - {f.name}")
    if len(npz_files) > 5:
        print(f"   ... y {len(npz_files) - 5} más")
    
    return npz_files


def load_batch_from_npz(npz_file):
    """
    Cargar un batch individual desde archivo .npz
    
    Args:
        npz_file (Path): Ruta al archivo .npz
        
    Returns:
        tuple: (images, labels) arrays de numpy
    """
    data = np.load(npz_file)
    
    # Intentar diferentes nombres de keys para imágenes
    images = None
    for key in ['images', 'X', 'data', 'x']:
        if key in data:
            images = data[key]
            break
    
    if images is None and len(data.keys()) > 0:
        keys = list(data.keys())
        images = data[keys[0]]
    
    # Intentar diferentes nombres para labels
    labels = None
    for key in ['labels', 'y', 'targets', 'label']:
        if key in data:
            labels = data[key]
            break
    
    if labels is None and len(data.keys()) > 1:
        keys = list(data.keys())
        labels = data[keys[1]]
    
    return images, labels

In [5]:
def preprocess_images_batch(images_batch):
    """
    Preprocesar batch de imágenes desde arrays .npz
    
    Realiza:
    1. Conversión de grayscale → RGB (si es necesario)
    2. Normalización del rango de valores
    3. Redimensionamiento a 224x224
    4. Preprocesamiento específico de MobileNetV2
    
    Args:
        images_batch (np.array): Batch de imágenes
        
    Returns:
        np.array: Batch procesado listo para el modelo
    """
    batch_processed = []
    
    for img_array in images_batch:
        try:
            # Si es grayscale (H, W), agregar dimensión de canal
            if img_array.ndim == 2:
                img_array = np.expand_dims(img_array, axis=-1)
            
            # Normalizar a [0, 255] si está en otro rango
            if img_array.max() <= 1.0:
                img_array = (img_array * 255).astype(np.uint8)
            else:
                img_array = img_array.astype(np.uint8)
            
            # Convertir a RGB si es grayscale (1 canal)
            if img_array.shape[-1] == 1:
                img_array = np.repeat(img_array, 3, axis=-1)
            
            # Asegurar que tiene exactamente 3 canales
            if img_array.shape[-1] != 3:
                img_array = img_array[:, :, :3]
            
            # Redimensionar a 224x224 si es necesario
            if img_array.shape[:2] != Config.IMG_SIZE:
                img_pil = Image.fromarray(img_array)
                img_pil = img_pil.resize(Config.IMG_SIZE, Image.LANCZOS)
                img_array = np.array(img_pil, dtype=np.float32)
            else:
                img_array = img_array.astype(np.float32)
            
            batch_processed.append(img_array)
            
        except Exception as e:
            print(f"⚠️ Error procesando imagen: {e}")
            batch_processed.append(
                np.zeros((224, 224, 3), dtype=np.float32)
            )
    
    # Convertir lista a array numpy
    batch_processed = np.array(batch_processed, dtype=np.float32)
    
    # Preprocesamiento específico de MobileNetV2
    batch_processed = preprocess_input(batch_processed)
    
    return batch_processed

In [6]:
def extract_embeddings_from_npz_files(model, npz_files, batch_size):
    """
    Extraer embeddings de múltiples archivos .npz
    
    Proceso:
    1. Itera sobre todos los archivos .npz
    2. Carga cada archivo
    3. Divide en mini-batches
    4. Extrae embeddings con MobileNetV2
    5. Libera memoria progresivamente
    
    Args:
        model: Modelo MobileNetV2 preentrenado
        npz_files (list): Lista de rutas a archivos .npz
        batch_size (int): Tamaño de batch para procesamiento
        
    Returns:
        tuple: (embeddings, labels) arrays de numpy
    """
    all_embeddings = []
    all_labels = []
    total_images_processed = 0
    
    print(f"\n🚀 Iniciando extracción de embeddings...")
    print(f"📦 Total archivos .npz: {len(npz_files)}")
    print(f"🔢 Batch size: {batch_size}\n")
    
    for npz_file in tqdm(npz_files, desc="Procesando archivos .npz"):
        # Cargar batch desde .npz
        images, labels = load_batch_from_npz(npz_file)
        
        if images is None:
            print(f"⚠️ Saltando {npz_file.name} (sin imágenes)")
            continue
        
        n_images = len(images)
        n_batches = (n_images + batch_size - 1) // batch_size
        
        # Dividir en mini-batches para procesamiento
        for i in range(n_batches):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, n_images)
            
            batch_images = images[start_idx:end_idx]
            
            # Preprocesar
            batch_processed = preprocess_images_batch(batch_images)
            
            # Extraer embeddings con MobileNetV2
            batch_embeddings = model.predict(batch_processed, verbose=0)
            
            all_embeddings.append(batch_embeddings)
            
            # Guardar labels si existen
            if labels is not None:
                batch_labels = labels[start_idx:end_idx]
                all_labels.extend(batch_labels)
            
            total_images_processed += len(batch_images)
            
            # Liberar memoria del mini-batch
            del batch_processed, batch_embeddings
        
        # Liberar memoria del archivo completo
        del images, labels
        gc.collect()
    
    # Concatenar todos los embeddings
    embeddings = np.concatenate(all_embeddings, axis=0)
    labels_array = (
        np.array(all_labels) if all_labels 
        else np.zeros(len(embeddings), dtype=int)
    )
    
    print("\n✅ Extracción completada!")
    print(f"📊 Total imágenes procesadas: {total_images_processed:,}")
    print(f"📐 Shape final embeddings: {embeddings.shape}")
    print(f"📐 Shape final labels: {labels_array.shape}")
    print(f"💾 Tamaño en memoria: {embeddings.nbytes / (1024**2):.2f} MB")
    
    return embeddings, labels_array

In [7]:
def apply_pca(embeddings, n_components=256):
    """
    Aplicar PCA para reducir dimensionalidad
    
    Args:
        embeddings (np.array): Embeddings originales
        n_components (int): Número de componentes finales
        
    Returns:
        tuple: (embeddings_pca, pca_model)
    """
    print("\n🔬 Aplicando PCA para reducir dimensionalidad...")
    print(f"📉 De {embeddings.shape[1]} → {n_components} dimensiones")
    
    pca = PCA(n_components=n_components)
    embeddings_pca = pca.fit_transform(embeddings)
    
    variance_explained = pca.explained_variance_ratio_.sum()
    size_reduction = (1 - embeddings_pca.nbytes / embeddings.nbytes) * 100
    
    print(f"✅ Dimensiones finales: {embeddings_pca.shape[1]}")
    print(f"📊 Varianza explicada: {variance_explained:.2%}")
    print(f"📉 Reducción de tamaño: {size_reduction:.1f}%")
    
    return embeddings_pca, pca

In [8]:
def save_embeddings_optimized(embeddings, labels, class_names, 
                               output_path, pca_model=None):
    """
    Guardar embeddings con optimizaciones aplicadas (sin gzip)
    
    Args:
        embeddings (np.array): Embeddings a guardar
        labels (np.array): Labels correspondientes
        class_names (list): Nombres de las clases
        output_path (str): Ruta completa del archivo de salida
        pca_model: Modelo PCA (opcional)
    """
    print("\n🔧 OPTIMIZANDO ARCHIVO PARA GITHUB")
    print("=" * 34)
    
    # Calcular tamaño original
    original_size_mb = embeddings.nbytes / (1024**2)
    print(f"\n📊 Tamaño original (float32): {original_size_mb:.2f} MB")
    
    # Optimización: Convertir a float16
    if Config.USE_FLOAT16:
        embeddings_optimized = embeddings.astype(np.float16)
        optimized_size_mb = embeddings_optimized.nbytes / (1024**2)
        reduction_1 = (1 - optimized_size_mb / original_size_mb) * 100
        print(f"✅ Convertido a float16: {optimized_size_mb:.2f} MB "
              f"(reducción: {reduction_1:.1f}%)")
    else:
        embeddings_optimized = embeddings
    
    # Optimizar labels
    labels_optimized = labels.astype(np.int16)
    print(f"✅ Labels optimizados a int16")
    
    # Crear diccionario de salida
    output_data = {
        'embeddings': embeddings_optimized,
        'labels': labels_optimized,
        'class_names': class_names,
        'model': 'MobileNetV2',
        'embedding_dim': embeddings_optimized.shape[1],
        'dtype': 'float16' if Config.USE_FLOAT16 else 'float32',
        'pca_applied': Config.USE_PCA
    }
    
    # Añadir modelo PCA si se usó
    if pca_model is not None:
        output_data['pca_model'] = pca_model
        print(f"✅ Modelo PCA incluido")
    
    # Guardar con pickle estándar (sin compresión gzip)
    print(f"\n💾 Guardando archivo embeddings.pkl...")
    
    with open(output_path, 'wb') as f:
        pickle.dump(output_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    # Calcular tamaños finales
    final_size_mb = os.path.getsize(output_path) / (1024**2)
    total_reduction = (1 - final_size_mb / original_size_mb) * 100
    
    print("\n" + "="*70)
    print("✅ ARCHIVO GUARDADO EXITOSAMENTE")
    print("="*70)
    print(f"📊 Tamaño original (float32): {original_size_mb:.2f} MB")
    print(f"📦 Tamaño final (optimizado): {final_size_mb:.2f} MB")
    print(f"🎯 Reducción total: {total_reduction:.1f}%")
    print(f"📂 Ubicación: {output_path}")
    
    github_ok = "SÍ ✅" if final_size_mb < 95 else "NO ❌"
    print(f"✅ Listo para GitHub (<100MB): {github_ok}")
    print("="*70)
    
    return final_size_mb

In [9]:
def generate_visualizations(embeddings, labels, class_names, output_path):
    """
    Generar visualizaciones de análisis de embeddings
    
    Args:
        embeddings (np.array): Embeddings extraídos
        labels (np.array): Labels correspondientes
        class_names (list): Nombres de las clases
        output_path (str): Carpeta donde guardar gráficos
    """
    print("\n📊 Generando visualizaciones...")
    
    # Figura 1: Distribución de embeddings (primeras 4 dimensiones)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(
        'Distribución de Embeddings MobileNetV2 (primeras 4 dimensiones)', 
        fontsize=16, fontweight='bold'
    )
    
    for idx, ax in enumerate(axes.flat):
        if idx < embeddings.shape[1]:
            ax.hist(embeddings[:, idx], bins=50, alpha=0.7, 
                    color='steelblue', edgecolor='black')
            ax.set_title(f'Dimensión {idx}', fontsize=12, fontweight='bold')
            ax.set_xlabel('Valor', fontsize=10)
            ax.set_ylabel('Frecuencia', fontsize=10)
            ax.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    viz_path_1 = os.path.join(output_path, 'embeddings_distribution.png')
    plt.savefig(viz_path_1, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Visualización 1 guardada: {viz_path_1}")
    
    # Figura 2: Distribución de clases
    unique_labels, counts = np.unique(labels, return_counts=True)
    
    if len(unique_labels) <= 30:  # Solo si hay pocas clases
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Ordenar por frecuencia
        sorted_indices = np.argsort(counts)[::-1]
        sorted_labels = unique_labels[sorted_indices]
        sorted_counts = counts[sorted_indices]
        
        # Nombres de clases para el eje X
        x_labels = [
            class_names[i] if i < len(class_names) else f"clase_{i}" 
            for i in sorted_labels
        ]
        
        ax.bar(range(len(sorted_counts)), sorted_counts, 
               color='coral', edgecolor='black', alpha=0.7)
        ax.set_xlabel('Clase', fontsize=12, fontweight='bold')
        ax.set_ylabel('Número de imágenes', fontsize=12, fontweight='bold')
        ax.set_title('Distribución de Imágenes por Clase', 
                     fontsize=14, fontweight='bold')
        ax.set_xticks(range(len(x_labels)))
        ax.set_xticklabels(x_labels, rotation=45, ha='right')
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        viz_path_2 = os.path.join(output_path, 'class_distribution.png')
        plt.savefig(viz_path_2, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"✅ Visualización 2 guardada: {viz_path_2}")

In [10]:
def print_usage_instructions():
    """Imprimir instrucciones de uso del archivo generado"""
    print("\n📖 CÓMO CARGAR EL ARCHIVO DESPUÉS:")
    print("-" * 35)
    print("""
import pickle
import numpy as np

# Cargar embeddings optimizados
with open('../backend/models/embeddings.pkl', 'rb') as f:
    data = pickle.load(f)

# Extraer datos (convertir float16 → float32 si es necesario)
embeddings = data['embeddings'].astype(np.float32)
labels = data['labels']
class_names = data['class_names']

print(f"Embeddings cargados: {embeddings.shape}")
print(f"Clases: {len(class_names)}")

# Si se usó PCA, el modelo también está disponible
if data['pca_applied']:
    pca_model = data['pca_model']
    print(f"PCA aplicado: {pca_model.n_components_} componentes")
    """)

In [11]:
def print_statistics(embeddings, labels, class_names):
    """
    Imprimir estadísticas finales del dataset
    
    Args:
        embeddings (np.array): Embeddings extraídos
        labels (np.array): Labels correspondientes
        class_names (list): Nombres de las clases
    """
    print("\n" + "="*70)
    print("📊 RESUMEN FINAL")
    print("="*70)
    
    print(f"\n🧠 Modelo utilizado: MobileNetV2")
    print(f"✅ Embeddings extraídos: {embeddings.shape[0]:,}")
    print(f"📏 Dimensiones por imagen: {embeddings.shape[1]}")
    print(f"🏷️ Número de clases: {len(class_names)}")
    
    print(f"\n📋 Clases detectadas: {class_names[:10]}")
    if len(class_names) > 10:
        print(f"    ... y {len(class_names) - 10} más")
    
    # Distribución por clase
    print("\n📊 Distribución de imágenes por clase:")
    unique_labels, counts = np.unique(labels, return_counts=True)
    
    # Mostrar las 10 clases más frecuentes
    sorted_indices = np.argsort(counts)[::-1][:10]
    for idx in sorted_indices:
        label_idx = unique_labels[idx]
        count = counts[idx]
        if label_idx < len(class_names):
            class_name = class_names[label_idx]
        else:
            class_name = f"clase_{label_idx}"
        percentage = count/len(labels)*100
        print(f"  {class_name}: {count:,} imágenes ({percentage:.1f}%)")
    
    # Estadísticas de embeddings
    print(f"\n📈 Estadísticas de embeddings:")
    print(f"  Media: {embeddings.mean():.4f}")
    print(f"  Desv. estándar: {embeddings.std():.4f}")
    print(f"  Mínimo: {embeddings.min():.4f}")
    print(f"  Máximo: {embeddings.max():.4f}")
    
    # Verificar que no hay NaN o Inf
    n_nan = np.isnan(embeddings).sum()
    n_inf = np.isinf(embeddings).sum()
    print(f"\n✅ Verificación de integridad:")
    print(f"  NaN values: {n_nan}")
    print(f"  Inf values: {n_inf}")
    
    if n_nan > 0 or n_inf > 0:
        print("  ⚠️ Se detectaron valores inválidos")
    else:
        print("  ✅ Todos los embeddings son válidos")

In [12]:
# =========================================================================
# 🚀 FUNCIÓN PRINCIPAL
# =========================================================================

def main():
    """Función principal del script"""
    
    try:
        # PASO 1: Cargar archivos del dataset
        npz_files = load_npz_files(Config.DATASET_PATH)
        
        # PASO 2: Analizar primer archivo para obtener metadatos
        print("\n🔍 Analizando primer archivo...")
        sample_images, sample_labels = load_batch_from_npz(npz_files[0])
        
        print(f"📐 Shape de imágenes: {sample_images.shape}")
        print(f"📐 Shape de labels: {sample_labels.shape if sample_labels is not None else 'N/A'}")
        print(f"📊 Tipo de dato: {sample_images.dtype}")
        print(f"📈 Rango de valores: [{sample_images.min():.2f}, {sample_images.max():.2f}]")
        
        # Detectar número de clases
        if sample_labels is not None:
            unique_labels = np.unique(sample_labels)
            n_classes = len(unique_labels)
            print(f"🏷️ Clases únicas detectadas: {n_classes}")
            
            # Generar nombres de clases
            if n_classes == 21:
                class_names = [
                    'pancakes', 'waffles', 'french_toast', 'eggs_benedict',
                    'scrambled_eggs', 'omelette', 'fried_eggs', 'bacon',
                    'sausage', 'hash_browns', 'toast', 'bagel',
                    'croissant', 'muffin', 'cereal', 'oatmeal',
                    'yogurt', 'fruit_salad', 'smoothie_bowl', 'avocado_toast',
                    'breakfast_burrito'
                ]
            else:
                class_names = [f"clase_{i}" for i in range(n_classes)]
        else:
            print("⚠️ No se encontraron labels")
            class_names = ["clase_0"]
            n_classes = 1
        
        print(f"📋 Nombres de clases: {class_names[:5]}{'...' if len(class_names) > 5 else ''}")
        
        # PASO 3: Cargar modelo MobileNetV2
        print("\n🔄 Cargando MobileNetV2 preentrenado...")
        base_model = MobileNetV2(
            weights='imagenet',
            include_top=False,
            pooling='avg',
            input_shape=(224, 224, 3)
        )
        print("✅ Modelo cargado correctamente!")
        print(f"📐 Salida del modelo: {base_model.output_shape}")
        print(f"💾 Parámetros totales: {base_model.count_params():,}")
        
        # PASO 4: Extraer embeddings
        embeddings, labels = extract_embeddings_from_npz_files(
            model=base_model,
            npz_files=npz_files,
            batch_size=Config.BATCH_SIZE
        )
        
        # PASO 5: Aplicar PCA si está activado
        pca_model = None
        if Config.USE_PCA:
            embeddings, pca_model = apply_pca(embeddings, Config.PCA_COMPONENTS)
        else:
            print("\n⚙️ PCA desactivado. Manteniendo dimensiones originales (1280).")
            print("💡 Para activar PCA y reducir ~80% más, cambia USE_PCA = True")
        
        # PASO 6: Guardar embeddings optimizados
        output_filepath = os.path.join(Config.OUTPUT_PATH, Config.OUTPUT_FILE)
        final_size = save_embeddings_optimized(
            embeddings=embeddings,
            labels=labels,
            class_names=class_names,
            output_path=output_filepath,
            pca_model=pca_model
        )
        
        # PASO 7: Generar visualizaciones (opcional)
        if Config.GENERATE_PLOTS:
            generate_visualizations(
                embeddings=embeddings,
                labels=labels,
                class_names=class_names,
                output_path=Config.OUTPUT_PATH
            )
        
        # PASO 8: Imprimir estadísticas finales
        print_statistics(embeddings, labels, class_names)
        
        # PASO 9: Imprimir instrucciones de uso
        print_usage_instructions()
        
        # Mensaje final
        print("\n" + "="*70)
        print("✨ ¡Todo listo! Puedes continuar con el entrenamiento de modelos.")
        print("="*70)
        
        # Verificar si es apto para GitHub
        if final_size > 95:
            print("\n⚠️ ADVERTENCIA: El archivo aún es >95MB")
            print("💡 Soluciones:")
            print("   1. Activa USE_PCA = True para reducir a ~15MB")
            print("   2. Usa Git LFS: https://git-lfs.github.com/")
            print("   3. Almacena en Google Drive/Dropbox")
        
        return 0
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return 1



In [13]:
# =========================================================================
# 🎬 PUNTO DE ENTRADA
# =========================================================================

if __name__ == "__main__":
    sys.exit(main())


📦 Archivos .npz encontrados: 1039
📋 Primeros archivos:
   - c000_batch0001.npz
   - c000_batch0002.npz
   - c000_batch0003.npz
   - c000_batch0004.npz
   - c000_batch0005.npz
   ... y 1034 más

🔍 Analizando primer archivo...
📐 Shape de imágenes: (20, 224, 224, 3)
📐 Shape de labels: (20,)
📊 Tipo de dato: float16
📈 Rango de valores: [0.00, 1.00]
🏷️ Clases únicas detectadas: 1
📋 Nombres de clases: ['clase_0']

🔄 Cargando MobileNetV2 preentrenado...
✅ Modelo cargado correctamente!
📐 Salida del modelo: (None, 1280)
💾 Parámetros totales: 2,257,984

🚀 Iniciando extracción de embeddings...
📦 Total archivos .npz: 1039
🔢 Batch size: 32



Procesando archivos .npz: 100%|██████████| 1039/1039 [14:19<00:00,  1.21it/s]



✅ Extracción completada!
📊 Total imágenes procesadas: 20,760
📐 Shape final embeddings: (20760, 1280)
📐 Shape final labels: (20760,)
💾 Tamaño en memoria: 101.37 MB

⚙️ PCA desactivado. Manteniendo dimensiones originales (1280).
💡 Para activar PCA y reducir ~80% más, cambia USE_PCA = True

🔧 OPTIMIZANDO ARCHIVO PARA GITHUB

📊 Tamaño original (float32): 101.37 MB
✅ Convertido a float16: 50.68 MB (reducción: 50.0%)
✅ Labels optimizados a int16

💾 Guardando archivo embeddings.pkl...

✅ ARCHIVO GUARDADO EXITOSAMENTE
📊 Tamaño original (float32): 101.37 MB
📦 Tamaño final (optimizado): 50.72 MB
🎯 Reducción total: 50.0%
📂 Ubicación: ../backend/models\embeddings.pkl
✅ Listo para GitHub (<100MB): SÍ ✅

📊 Generando visualizaciones...
✅ Visualización 1 guardada: ../backend/models\embeddings_distribution.png
✅ Visualización 2 guardada: ../backend/models\class_distribution.png

📊 RESUMEN FINAL

🧠 Modelo utilizado: MobileNetV2
✅ Embeddings extraídos: 20,760
📏 Dimensiones por imagen: 1280
🏷️ Número de 

SystemExit: 0